# Ragas를 활용한 합성 테스트 데이터 생성
## 검색 대상 문서 로드

In [1]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str)->bool:
    return file_path.endswith(".md")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter
)

documents = loader.load()
print(len(documents))

28


## Ragas를 활용한 합성 테스트 데이터 생성 구현


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas.utils import num_tokens_from_string


# 1. RAGAS용 LLM / Embedding 설정
generator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-5.4-mini")
)

generator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)


# 2. RAGAS 0.4.3 default transform 에러 방지를 위해 짧은 문서 제외
documents_for_testset = [
    doc for doc in documents
    if num_tokens_from_string(doc.page_content) > 500
]

print(f"원본 문서 수: {len(documents)}")
print(f"테스트셋 생성에 사용할 문서 수: {len(documents_for_testset)}")

if not documents_for_testset:
    raise ValueError(
        "500 토큰보다 긴 문서가 없습니다. 더 긴 문서를 사용하거나 chunk 크기를 키워주세요."
    )


# 3. 예전 distributions 대응
# simple        -> SingleHopSpecificQuerySynthesizer
# reasoning     -> MultiHopAbstractQuerySynthesizer
# multi_context -> MultiHopSpecificQuerySynthesizer
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]


# 4. Testset 생성
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

testset = generator.generate_with_langchain_docs(
    documents_for_testset,
    testset_size=4,
    query_distribution=query_distribution,
)


# 5. Pandas DataFrame으로 확인
testset.to_pandas()

/var/folders/hm/gd5jtc_93xn25x30bh0p66jw0000gn/T/ipykernel_96876/2321339401.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(
/var/folders/hm/gd5jtc_93xn25x30bh0p66jw0000gn/T/ipykernel_96876/2321339401.py:22: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(


원본 문서 수: 28
테스트셋 생성에 사용할 문서 수: 10


Applying SummaryExtractor:   8%|▊         | 1/13 [00:02<00:24,  2.04s/it]Property 'summary' already exists in node '73d0e6'. Skipping!
Property 'summary' already exists in node '80d805'. Skipping!
Applying EmbeddingExtractor:   0%|          | 0/13 [00:00<?, ?it/s]Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.6/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108d6b080> is already entered
Applying EmbeddingExtractor:   8%|▊         | 1/13 [00:01<00:15,  1.29s/it]Property 'summary_embedding' already exists in node '73d0e6'. Skipping!
Property 'summary_embedding' already exists in node '80d805'. Skipping!
Applying ThemesExtractor:   0%|          | 0/20 [00:00<?, ?it/s]Task wa